# Customer Segmentation — RFM + K-Means

Segments e-commerce customers using **Recency**, **Frequency**, **Monetary** (RFM) features and K-Means clustering.

**Dataset:** [Online Retail II](https://www.kaggle.com/datasets/mashlyn/online-retail-ii-uci) — place the file at `data/online_retail_II.xlsx`

## 1. Load & Explore

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_excel("data/online_retail_II.xlsx")
print(df.shape)
print(df.head())
print(df.dtypes)

## 2. Data Cleaning

In [ ]:
# Remove records with missing Customer IDs (can't segment anonymous visitors)
df = df[df['Customer ID'].notna()]

# Remove negative/zero prices (data entry errors or test records)
df = df[df['Price'] >= 0]

print(df.shape)
print(df.isnull().sum())
print(f"Zero or negative prices remaining: {(df['Price'] == 0).sum():,}")

## 3. Feature Engineering

In [ ]:
# Revenue per line item
df['Revenue'] = df['Quantity'] * df['Price']

# Flag returns (invoices starting with 'C' are cancellations)
df['Invoice'] = df['Invoice'].astype(str)
df['Return'] = np.where(df['Invoice'].str.startswith('C'), 1, 0)

In [ ]:
# Aggregate to customer level: Revenue, Frequency, Returns, date range
customer_df = df.groupby('Customer ID').agg(
    Revenue=('Revenue', 'sum'),
    MinInvoiceDate=('InvoiceDate', 'min'),
    MaxInvoiceDate=('InvoiceDate', 'max'),
    Invoice=('Invoice', 'nunique'),
    Returns=('Return', 'sum')
).reset_index().sort_values('Revenue', ascending=False)

customer_df

In [ ]:
# Loyalty = number of days between first and last order
customer_df['loyalty'] = (customer_df['MaxInvoiceDate'] - customer_df['MinInvoiceDate']).dt.days

# Recency = days since last order (lower = more recent)
reference_date = df['InvoiceDate'].max()
customer_df['Recency'] = (reference_date - customer_df['MaxInvoiceDate']).dt.days
customer_df.rename(columns={'Invoice': 'Frequency'}, inplace=True)

customer_df

## 4. Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler

df_classification = customer_df.drop(columns=['MinInvoiceDate', 'MaxInvoiceDate'])
print(df_classification.columns)

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(df_classification.drop(columns=['Customer ID']))

## 5. Find Optimal Number of Clusters (Elbow Method)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

inertia = []
for k in range(1, 11):
    model = KMeans(n_clusters=k, random_state=42)
    model.fit(rfm_scaled)
    inertia.append(model.inertia_)

plt.plot(range(1, 11), inertia, marker='o')
plt.title('Elbow Method — Optimal K')
plt.xlabel('Number of clusters')
plt.ylabel('Inertia')
plt.show()

## 6. Train Final K-Means Model

In [ ]:
# Train with 4 clusters (identified from elbow plot)
model = KMeans(n_clusters=4, random_state=42).fit(rfm_scaled)
customer_df['Cluster'] = model.labels_
customer_df['Cluster'].value_counts()

## 7. Visualise Segments

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for ax, x_col in zip(axes, ['Recency', 'Frequency', 'loyalty']):
    sc = ax.scatter(customer_df[x_col], customer_df['Revenue'],
                    c=customer_df['Cluster'], cmap='viridis', alpha=0.7)
    ax.set_xlabel(x_col)
    ax.set_ylabel('Revenue')
    ax.set_title(f'Revenue vs {x_col}')
    plt.colorbar(sc, ax=ax)

plt.tight_layout()
plt.show()

In [ ]:
# Average metrics per cluster to interpret the segments
customer_df.groupby('Cluster').agg({
    'Revenue': 'mean',
    'Frequency': 'mean',
    'Recency': 'mean',
    'loyalty': 'mean'
}).round(1)

## 8. Label Clusters

In [ ]:
cluster_names = {
    0: 'Loyal regulars',
    1: 'Lost',
    2: 'Champions',
    3: 'At risk'
}

customer_df['Cluster Name'] = customer_df['Cluster'].map(cluster_names)
customer_df

In [ ]:
# Save segmented customers for use in churn prediction notebook
customer_df.to_csv('data/customer_segments.csv', index=False)
print("Saved to data/customer_segments.csv")